# D2: block tables, slot mapping and paged KV cache

**Goal:** several cached requests use a shared pool of fixed-size KV blocks. Their logical token order must not depend on physical block order.

**Spec:** ordered and scrambled storage match separate ours/HF; independent request lengths support mixed prefill/decode; unknown IDs, overflow, invalid append positions and shared blocks raise. Existing no-cache/contiguous behavior stays supported.

This lesson uses manually assigned **disjoint blocks** and readable PyTorch gathers. No allocator, prefix sharing, scheduler or fast kernel yet. D3 adds block management.

Run top to bottom on a **fresh Colab T4**. Two FP32 model copies need about 8.8 GB of weights. Runtime validation is pending; local checks are static only. Save previous notebook/results before replacing the runtime.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], check=True, capture_output=True, text=True).stdout)


## 1. Test first

Run B8/C2/C3/D1/D2 tests. D2 checks block sizes 1-4, scrambled physical order, MHA/GQA/MQA, boundary-crossing chunks, mixed prefill/decode, interleaved routing, failure/retry, reset and ownership errors.

A failed test stops the lesson. The C2 two-GPU test can skip on a single T4. CUDA tests should run here.

In [ ]:
import gc
import torch
from tinyserve.basics import load_model, encode
from tinyserve.models.llama import LlamaForCausalLM
from tinyserve.models.attention import AttentionMetadata
from tinyserve.probe import compare_paged_batch
from tinyserve.envinfo import collect_env, save_result

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun.")
DEVICE = "cuda"
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
print(collect_env())
test_run = subprocess.run([sys.executable, "-m", "pytest", "tests/test_b8_model.py",
                           "tests/test_c2_generation.py", "tests/test_c3_cache.py",
                           "tests/test_d1_batching.py", "tests/test_d2_paged_cache.py", "-v", "-ra"],
                          capture_output=True, text=True)
print(test_run.stdout)
print(test_run.stderr)
test_run.check_returncode()


## 2. Logical position is not physical slot

For block size B=2 and block table `[4, 1, 5]`:

```
logical positions: 0  1  2  3   4   5
logical blocks:    0  0  1  1   2   2
offsets:           0  1  0  1   0   1
physical blocks:   4  4  1  1   5   5
physical slots:    8  9  2  3  10  11
```

`physical_slot = table[position // B] * B + position % B`.

Logical order is 0,1,2,... even when physical addresses jump backward. Sorting the table is wrong. RoPE always uses logical position, not physical slot.

Load the real FP32 model, then inspect this addressing without running attention. The pool stores `(22, blocks, B, 4, 64)` for each of K and V.

In [ ]:
tokenizer, hf_model = load_model(device=DEVICE, dtype=torch.float32)
hf_model.set_attn_implementation("eager")
our_model = LlamaForCausalLM.from_huggingface(hf_model)
hand_pool = our_model.new_paged_cache(6, 2, {7: [4, 1, 5], 9: [2, 0]})
hand_positions = torch.tensor([0, 1, 2, 3, 4, 5, 0, 1, 2, 3], device=DEVICE)
hand_owners = hand_positions.new_tensor([7] * 6 + [9] * 4)
hand_slots = hand_pool.slot_mapping(hand_positions, hand_owners)
assert hand_slots.tolist() == [8, 9, 2, 3, 10, 11, 4, 5, 0, 1]
hand_addressing = {"block_size": 2, "tables": {str(key): list(value) for key, value in hand_pool.block_tables.items()},
                   "positions": hand_positions.tolist(), "request_ids": hand_owners.tolist(),
                   "slots": hand_slots.tolist(), "pool_shape": list(hand_pool.key.shape),
                   "allocated_bytes": hand_pool.allocated_bytes}
print(hand_addressing)
del hand_pool


## 3. Create ordered and scrambled pools

Each request gets enough blocks for its prompt plus two processed continuation tokens. All block tables are disjoint. The whole pool is preallocated; table assignment is not an allocator.

Create two layouts for the same requests: sequential physical IDs and globally reversed physical IDs. They must produce the same token choices and agree with our no-cache and HF references.

Fill storage with NaN first. Correct attention must never read unassigned or unwritten slots. Only committed prefix plus the current append chunk is gathered.

In [ ]:
prompt_texts = ["The capital of France is", "The capital of Germany is",
                "A tiny language model can help people by"]
request_prefixes = {index: encode(tokenizer, text, DEVICE) for index, text in enumerate(prompt_texts)}
BLOCK_SIZE = 4
block_counts = {request_id: (ids.numel() + 2 + BLOCK_SIZE - 1) // BLOCK_SIZE
                for request_id, ids in request_prefixes.items()}
NUM_BLOCKS = sum(block_counts.values())
ordered_tables, scrambled_tables = {}, {}
cursor = 0
for request_id, count in block_counts.items():
    ordered_tables[request_id] = list(range(cursor, cursor + count))
    scrambled_tables[request_id] = [NUM_BLOCKS - 1 - block for block in ordered_tables[request_id]]
    cursor += count
ordered_pool = our_model.new_paged_cache(NUM_BLOCKS, BLOCK_SIZE, ordered_tables)
scrambled_pool = our_model.new_paged_cache(NUM_BLOCKS, BLOCK_SIZE, scrambled_tables)
for pool in (ordered_pool, scrambled_pool):
    pool.key.fill_(float("nan"))
    pool.value.fill_(float("nan"))
print("ordered:", ordered_tables)
print("scrambled:", scrambled_tables)
print("pool shape:", tuple(ordered_pool.key.shape), "bytes per pool:", ordered_pool.allocated_bytes)
stage_results = []

def run_stage(label, active_requests):
    ordered = compare_paged_batch(our_model, hf_model, ordered_pool, active_requests)
    scrambled = compare_paged_batch(our_model, hf_model, scrambled_pool, active_requests)
    assert [row["next_token_id"] for row in ordered["requests"]] == [row["next_token_id"] for row in scrambled["requests"]]
    assert dict(ordered_pool.lengths) == dict(scrambled_pool.lengths)
    stage_results.append({"label": label, "ordered": ordered, "scrambled": scrambled,
                          "all_request_lengths": dict(ordered_pool.lengths)})
    print(label, "positions:", scrambled["positions"], "request IDs:", scrambled["sequence_ids"],
          "physical slots:", scrambled["slot_mapping"])
    for row in scrambled["requests"]:
        print(f"request={row['request_id']} start={row['start_position']} new={row['new_tokens']} "
              f"length={row['committed_length']} diff(no-cache)={row['paged_vs_no_cache_max_abs_diff']:.2e} "
              f"diff(HF)={row['paged_vs_hf_max_abs_diff']:.2e}")
    return {row["request_id"]: row["next_token_id"] for row in ordered["requests"]}


## 4. Prefill two requests; keep the third pending

The verification helper receives complete request prefixes, but forwards only suffix tokens not yet committed. For new requests, the suffix is the full prompt.

A model forward produces the next-token logits but does not process that predicted token. Its K/V are appended only when we feed it into the next forward. This is the same C3 length distinction, now independently per request.

In [ ]:
next_ids = run_stage("initial prefill", {0: request_prefixes[0], 1: request_prefixes[1]})
assert ordered_pool.lengths[2] == 0
for request_id, token_id in next_ids.items():
    print("request", request_id, "predicted", repr(tokenizer.decode([token_id])))


## 5. Mix one decode token with a new prefill

Append request 0's predicted token to its prefix. Admit request 2's whole prompt. Pause request 1: it contributes no rows and keeps its committed length.

The batch contains one request's absolute decode position and another's positions starting at zero. Different lengths do not require padding. Each layer gathers only the owning request's blocks.

These are fixed verification forwards, not a scheduler or an EOS-aware multi-request generation loop.

In [ ]:
paused_length = ordered_pool.lengths[1]
request_prefixes[0] = torch.cat((request_prefixes[0], request_prefixes[0].new_tensor([next_ids[0]])))
mixed_next_ids = run_stage("decode 0 + prefill 2; pause 1", {0: request_prefixes[0], 2: request_prefixes[2]})
assert ordered_pool.lengths[1] == paused_length
next_ids.update(mixed_next_ids)


## 6. Resume all requests in a different flat order

Feed one previously predicted token for each request, ordered `[2, 1, 0]`. Positions continue from each request's own length; physical addresses still come from its table. Scatter must return logits in the incoming flat order.

Crossing a logical block boundary can jump to an unrelated physical block. The scrambled pool exercises that addressing while HF checks the mathematical answer.

In [ ]:
for request_id in (2, 1, 0):
    request_prefixes[request_id] = torch.cat((request_prefixes[request_id], request_prefixes[request_id].new_tensor([next_ids[request_id]])))
final_next_ids = run_stage("decode all; reordered 2,1,0", {request_id: request_prefixes[request_id] for request_id in (2, 1, 0)})
final_lengths = dict(ordered_pool.lengths)
print("final independent lengths:", final_lengths)
print("next tokens:", {request_id: tokenizer.decode([token_id]) for request_id, token_id in final_next_ids.items()})
del hf_model
gc.collect()
torch.cuda.empty_cache()


## 7. Explicit boundaries and reset

D2 rejects overlapping blocks: sharing without ownership/refcounts/copy-on-write could corrupt requests. Pool capacity and model context limits are separate checks. Append positions cannot restart at zero unless that request is reset.

`reset(0)` forgets only request 0, reusing its assigned blocks without freeing them. It leaves the other requests intact. D3 will add a free list and ownership management.

Reset/recreate after changing weights or prompts; recreate after model device/dtype changes. A paged cache is inference state for one model and must not be used concurrently.

In [ ]:
try:
    our_model.new_paged_cache(2, 4, {0: [0], 1: [0]})
except ValueError as error:
    overlap_error = str(error)
    print("expected overlap error:", overlap_error)
else:
    raise AssertionError("D2 must reject shared blocks")
try:
    with torch.no_grad():
        ids = request_prefixes[0][:1]
        positions = torch.zeros_like(ids)
        owners = torch.zeros_like(ids)
        our_model(ids, positions, AttentionMetadata(positions, owners, paged_cache=scrambled_pool))
except ValueError as error:
    position_error = str(error)
    print("expected append error:", position_error)
else:
    raise AssertionError("committed request cannot restart positions")
pool_pointer = scrambled_pool.key.data_ptr()
scrambled_pool.reset(0)
assert scrambled_pool.key.data_ptr() == pool_pointer and scrambled_pool.lengths[0] == 0
assert scrambled_pool.lengths[1] == final_lengths[1] and scrambled_pool.lengths[2] == final_lengths[2]
with torch.no_grad():
    reset_ids = request_prefixes[0][:2]
    reset_positions = torch.arange(reset_ids.numel(), device=reset_ids.device)
    reset_owners = torch.zeros_like(reset_ids)
    reset_output = our_model(reset_ids, reset_positions, AttentionMetadata(reset_positions, reset_owners, paged_cache=scrambled_pool))
    reference_output = our_model(reset_ids, reset_positions, AttentionMetadata(reset_positions))
torch.testing.assert_close(reset_output, reference_output, rtol=1e-5, atol=1e-4)
reset_diff = (reset_output - reference_output).abs().max().item()
print("reset parity difference:", reset_diff)


## 8. Save evidence and interview questions

1. **What does a block table translate?** Logical block number to physical block ID; offset stays the same.
2. **Why scramble physical order?** Correctness must not assume contiguous or sorted placement.
3. **Why reject shared blocks?** Independent writes need ownership and copy-on-write rules not implemented here.

Gathered K/V are temporary copies; this is not a high-performance kernel. The ranges below are numerical correctness differences, not timing measurements.

Save the JSON and notebook with outputs before replacing the runtime. **Next after D2 validation: D3, block manager free list and reference counts.**

In [ ]:
all_rows = [row for stage in stage_results for layout in ("ordered", "scrambled") for row in stage[layout]["requests"]]
no_cache_diffs = [row["paged_vs_no_cache_max_abs_diff"] for row in all_rows]
hf_diffs = [row["paged_vs_hf_max_abs_diff"] for row in all_rows]
saved_path = save_result("D2_paged_cache", {
    "dtype": "float32", "backend": "pytorch_paged_gather", "manual_disjoint_blocks": True,
    "hand_addressing": hand_addressing, "block_size": BLOCK_SIZE, "num_blocks": NUM_BLOCKS,
    "ordered_tables": ordered_tables, "scrambled_tables": scrambled_tables,
    "pool_allocated_bytes": ordered_pool.allocated_bytes, "stages": stage_results,
    "final_lengths_before_reset": final_lengths,
    "paged_vs_no_cache_diff_range": {"min": min(no_cache_diffs), "max": max(no_cache_diffs)},
    "paged_vs_hf_diff_range": {"min": min(hf_diffs), "max": max(hf_diffs)},
    "overlap_error": overlap_error, "invalid_append_error": position_error,
    "reset_reused_pool_storage": True, "reset_max_abs_diff": reset_diff,
    "pytest": {"returncode": test_run.returncode, "stdout": test_run.stdout, "stderr": test_run.stderr},
})
print(saved_path)
print(saved_path.read_text())


In [ ]:
del ordered_pool, scrambled_pool, pool, reset_output, reference_output, our_model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory still held by this kernel: {torch.cuda.memory_allocated() / 2**20:.0f} MiB")
